# 🏪 Superstore Sales Analytics Pipeline
## Real-World Revision Project — Days 1, 2 & 3

---

### 📂 Dataset: Global Superstore (Kaggle)

**Download it before running this notebook:**

**Option A — Kaggle Web UI (easiest):**
1. Go to → https://www.kaggle.com/datasets/vivek468/superstore-dataset-final
2. Click **Download** → unzip → you get `Sample - Superstore.csv`
3. Place it in the same folder as this notebook
4. Update `FILE_PATH` in Section 1 if your filename differs

**Option B — Kaggle CLI:**
```bash
pip install kaggle
kaggle datasets download -d vivek468/superstore-dataset-final
unzip superstore-dataset-final.zip
```

**Dataset at a glance:**

| Column | Type | Description |
|--------|------|-------------|
| Row ID | int | Row number |
| Order ID | str | Unique order identifier |
| Order Date | str | Date order was placed |
| Ship Date | str | Date order was shipped |
| Ship Mode | str | Shipping class |
| Customer ID | str | Unique customer ID |
| Customer Name | str | Full name |
| Segment | str | Consumer / Corporate / Home Office |
| Country | str | Country of delivery |
| City | str | City of delivery |
| State | str | State of delivery |
| Region | str | East / West / Central / South |
| Product ID | str | Unique product ID |
| Category | str | Furniture / Office Supplies / Technology |
| Sub-Category | str | e.g. Chairs, Phones, Binders |
| Product Name | str | Full product name |
| Sales | float | Revenue for the line item |
| Quantity | int | Units ordered |
| Discount | float | Discount fraction (0.0 – 0.8) |
| Profit | float | Profit (can be negative!) |

~9,994 rows. No login required after download.

---

### 🎯 What We're Doing

| Section | Concepts Revised |
|---------|------------------|
| 1. Setup & Load | Day 1: Variables, File I/O, Exception Handling |
| 2. Exploration | Day 2: DataFrame inspection, dtypes, describe |
| 3. Cleaning | Day 2: Nulls, Duplicates, Type Conversion + Day 3: `re` |
| 4. Feature Engineering | Day 1: Lambda, Comprehensions + Day 2: DateTime + Day 3: Walrus |
| 5. Analysis | Day 2: GroupBy, Merge, Pivot, Crosstab + Day 3: Counter |
| 6. OOP Pipeline | Day 1: Classes, Inheritance + Day 3: Decorators, `logging`, `@property` |
| 7. Visualisation | Day 3: Matplotlib — 5 chart types |
| 8. Export | Day 2: CSV / Excel / Parquet + Day 3: Generators |

---

## 📦 Step 0 — Imports
Run this cell first. Install any missing packages with `pip install <name>`.

In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────────
import re
import sys
import time
import copy
import json
import logging
import itertools
from pathlib import Path
from datetime import datetime
from functools import wraps
from collections import Counter, defaultdict

# ── Third-Party ───────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Settings ──────────────────────────────────────────────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:,.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')

print(f"✅ All imports successful!")
print(f"   Pandas  : {pd.__version__}")
print(f"   NumPy   : {np.__version__}")
print(f"   Python  : {sys.version.split()[0]}")

---
## 🔵 Section 1 — Setup & Load Raw Data (Day 1)

### Concepts Covered
- **Variables & Data Types** — Path, strings, constants
- **Tuples** — immutable column name collections
- **Dictionaries** — column rename maps
- **Exception Handling** — `try/except` for safe file loading
- **`pathlib`** — modern file path handling

In [ ]:
# ── 1.1 Configuration — Variables & Data Types (Day 1) ───────────────────────

# String variable — update this if your CSV has a different filename
FILE_PATH = Path('Sample - Superstore.csv')   # pathlib Path object

# Tuple — immutable; great for column groups you won't accidentally change
DATE_COLS    = ('Order Date', 'Ship Date')       # columns to parse as datetime
NUMERIC_COLS = ('Sales', 'Quantity', 'Discount', 'Profit')  # expected numeric

# Dictionary — maps original column names to cleaner Python-friendly names
COL_RENAME = {
    'Row ID'       : 'row_id',
    'Order ID'     : 'order_id',
    'Order Date'   : 'order_date',
    'Ship Date'    : 'ship_date',
    'Ship Mode'    : 'ship_mode',
    'Customer ID'  : 'customer_id',
    'Customer Name': 'customer_name',
    'Segment'      : 'segment',
    'Country'      : 'country',
    'City'         : 'city',
    'State'        : 'state',
    'Postal Code'  : 'postal_code',
    'Region'       : 'region',
    'Product ID'   : 'product_id',
    'Category'     : 'category',
    'Sub-Category' : 'sub_category',
    'Product Name' : 'product_name',
    'Sales'        : 'sales',
    'Quantity'     : 'quantity',
    'Discount'     : 'discount',
    'Profit'       : 'profit',
}

# Output directory using pathlib
OUTPUT_DIR = Path('superstore_output')
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"📁 File to load : {FILE_PATH}")
print(f"📁 Output folder: {OUTPUT_DIR.resolve()}")
print(f"🗂️  Date columns  : {DATE_COLS}")
print(f"🔢 Numeric cols  : {NUMERIC_COLS}")

In [ ]:
# ── 1.2 Safe File Loading with Exception Handling (Day 1) ────────────────────

# Custom exception class (Day 1 — Custom Exceptions)
class DataLoadError(Exception):
    """Raised when the source data file cannot be loaded."""
    def __init__(self, path, message="Could not load data file"):
        self.path    = path
        self.message = message
        super().__init__(f"{message}: {path}")


def load_superstore(filepath: Path) -> pd.DataFrame:
    """
    Load the Kaggle Superstore CSV.
    Tries Latin-1 encoding if UTF-8 fails (common with Kaggle CSVs).
    Raises DataLoadError if file is missing or completely unreadable.
    """
    if not filepath.exists():
        raise DataLoadError(filepath, "File not found. Did you download it from Kaggle?")

    for encoding in ['utf-8', 'latin-1', 'cp1252']:     # try encodings in order
        try:
            df = pd.read_csv(filepath, encoding=encoding)
            print(f"✅ Loaded with encoding: {encoding}")
            return df
        except UnicodeDecodeError:
            continue
        except Exception as e:
            raise DataLoadError(filepath, str(e)) from e

    raise DataLoadError(filepath, "All encodings failed")


# Load!
try:
    df_raw = load_superstore(FILE_PATH)
    print(f"📊 Shape         : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
    print(f"📋 Columns       : {list(df_raw.columns)}")
except DataLoadError as e:
    print(f"\n❌ {e}")
    print("   ➡️  Download from: https://www.kaggle.com/datasets/vivek468/superstore-dataset-final")

---
## 🟡 Section 2 — Explore the Raw Data (Day 2)

### Concepts Covered
- **DataFrame inspection** — `.head()`, `.info()`, `.describe()`
- **dtypes** — understanding what Pandas inferred vs what we need
- **Selecting columns** — single column, multiple columns
- **`loc` and `iloc`** — label vs position based indexing
- **Sorting** — by single and multiple columns

In [ ]:
# ── 2.1 First Look at the Data ────────────────────────────────────────────────

print("=" * 60)
print("FIRST 5 ROWS")
print("=" * 60)
print(df_raw.head())

print("\n" + "=" * 60)
print("DATA TYPES & NON-NULL COUNTS")
print("=" * 60)
print(df_raw.info())

print("\n" + "=" * 60)
print("SUMMARY STATISTICS (numeric columns only)")
print("=" * 60)
print(df_raw.describe())

In [ ]:
# ── 2.2 Selecting Columns (Day 2 — loc / iloc) ───────────────────────────────

# Single column → Series
print("Sales column (first 5 values):")
print(df_raw['Sales'].head())

# Multiple columns → DataFrame
print("\nOrder info columns:")
print(df_raw[['Order ID', 'Customer Name', 'Category', 'Sales', 'Profit']].head())

# iloc — integer position (first 3 rows, first 5 columns)
print("\niLoc [0:3, 0:5]:")
print(df_raw.iloc[0:3, 0:5])

# loc — label based (rows 0-2, named columns)
print("\nLoc [0:2, specific columns]:")
print(df_raw.loc[0:2, ['Order ID', 'Order Date', 'Sales', 'Profit']])

In [ ]:
# ── 2.3 Sorting & Filtering (Day 2) ──────────────────────────────────────────

# Top 10 orders by Sales
print("Top 10 orders by Sales:")
print(df_raw.sort_values('Sales', ascending=False)[['Order ID','Customer Name','Category','Sales','Profit']].head(10))

# Filter: only Technology category
tech = df_raw[df_raw['Category'] == 'Technology']
print(f"\nTechnology orders: {len(tech):,} rows")

# Filter: Loss-making orders (negative profit)
losses = df_raw[df_raw['Profit'] < 0]
print(f"Loss-making orders: {len(losses):,} rows  ({len(losses)/len(df_raw)*100:.1f}% of all orders)")

# Filter: High discount AND negative profit (isin example too)
bad_deals = df_raw[(df_raw['Discount'] >= 0.4) & (df_raw['Profit'] < 0)]
print(f"\nHigh-discount loss orders (discount ≥ 40%): {len(bad_deals):,} rows")

# isin example — specific regions
east_west = df_raw[df_raw['Region'].isin(['East', 'West'])]
print(f"East + West orders: {len(east_west):,} rows")

---
## 🟢 Section 3 — Data Cleaning (Day 2 + Day 3: `re` module)

### Concepts Covered
- **Rename columns** — clean Python-style names
- **Null / Missing values** — detect and handle
- **Duplicate detection & removal**
- **Type conversion** — `astype`, `pd.to_datetime`, `pd.to_numeric`
- **Day 3 `re` module** — clean messy Product IDs with regex

In [ ]:
# ── 3.1 Rename Columns ────────────────────────────────────────────────────────

df = df_raw.rename(columns=COL_RENAME)
print("Columns after rename:")
print(list(df.columns))

In [ ]:
# ── 3.2 Null Value Analysis ───────────────────────────────────────────────────

null_counts  = df.isnull().sum()
null_pct     = (df.isnull().mean() * 100).round(2)
null_report  = pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})

print("🔍 Null Value Report:")
print(null_report[null_report['null_count'] > 0])

if null_report['null_count'].sum() == 0:
    print("   ✅ No null values — this is a clean Kaggle dataset!")
    print("   (In real-world pipelines you would almost always see nulls here.)")

In [ ]:
# ── 3.3 Duplicate Detection ───────────────────────────────────────────────────

n_full_dupes = df.duplicated().sum()
print(f"Exact duplicate rows        : {n_full_dupes}")

# Superstore can have duplicate Order IDs because one order = multiple products
n_order_dupes = df.duplicated(subset=['order_id', 'product_id']).sum()
print(f"Duplicate Order+Product pairs: {n_order_dupes}")

# Show how many unique orders vs rows
print(f"\nTotal rows        : {len(df):,}")
print(f"Unique order IDs  : {df['order_id'].nunique():,}")
print(f"Unique products   : {df['product_id'].nunique():,}")
print(f"Unique customers  : {df['customer_id'].nunique():,}")

# Drop any true full duplicates if present
before = len(df)
df = df.drop_duplicates()
print(f"\nRows after dedup: {len(df):,}  (removed {before - len(df)})")

In [ ]:
# ── 3.4 Data Type Conversion (Day 2) ──────────────────────────────────────────

print("Data types BEFORE conversion:")
print(df[['order_date', 'ship_date', 'sales', 'quantity', 'discount', 'profit']].dtypes)
print()

# Convert date columns (Kaggle Superstore format: 'MM/DD/YYYY' or 'DD-MM-YYYY')
for col in ['order_date', 'ship_date']:
    df[col] = pd.to_datetime(df[col], infer_datetime_format=True, errors='coerce')

# Ensure numeric types are correct (safe conversion)
df['sales']    = pd.to_numeric(df['sales'],    errors='coerce')
df['profit']   = pd.to_numeric(df['profit'],   errors='coerce')
df['discount'] = pd.to_numeric(df['discount'], errors='coerce')
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce').astype('Int64')  # nullable int

# postal_code should stay as string (leading zeros get lost if cast to int)
df['postal_code'] = df['postal_code'].astype(str)

print("Data types AFTER conversion:")
print(df[['order_date', 'ship_date', 'sales', 'quantity', 'discount', 'profit']].dtypes)

In [ ]:
# ── 3.5 Day 3: `re` Module — Validate & Clean Product IDs ─────────────────────
#
# Superstore product_id format: FUR-BO-10001798
# (Category prefix)-(Sub-cat prefix)-(numeric suffix)
# We'll use regex to:
#   1) Validate the format
#   2) Extract the numeric part
#   3) Flag any IDs that don't match (anomalies)

PRODUCT_ID_PATTERN = re.compile(r'^[A-Z]{2,3}-[A-Z]{2,3}-\d+$')

def validate_product_id(pid: str) -> bool:
    """Returns True if product_id matches expected pattern."""
    return bool(PRODUCT_ID_PATTERN.match(str(pid)))

def extract_numeric_id(pid: str) -> str:
    """Extract just the numeric suffix from a product ID."""
    match = re.search(r'(\d+)$', str(pid))    # find digits at end
    return match.group(1) if match else None

# Apply regex functions to the DataFrame
df['product_id_valid']   = df['product_id'].apply(validate_product_id)
df['product_id_numeric'] = df['product_id'].apply(extract_numeric_id)

invalid = df[~df['product_id_valid']]
print(f"✅ Valid product IDs   : {df['product_id_valid'].sum():,}")
print(f"❌ Invalid product IDs : {len(invalid)}")

# Sample regex output
print("\nSample product IDs with extracted numeric part:")
print(df[['product_id', 'product_id_valid', 'product_id_numeric']].head(8))

# Also use re.findall to count how many products per category prefix
prefixes = df['product_id'].str.extract(r'^([A-Z]{2,3})-')
print(f"\nProduct category prefix counts:")
print(prefixes[0].value_counts())

---
## 🟠 Section 4 — Feature Engineering (Day 1 + Day 2 DateTime + Day 3 Walrus)

### Concepts Covered
- **Lambda functions** — quick column transforms
- **List comprehensions** — Pythonic column creation
- **Dictionary comprehensions** — lookup tables
- **DateTime features** — `.dt.year`, `.dt.month_name()`, `.dt.quarter`
- **Walrus operator `:=`** (Day 3) — assign and check in one step

In [ ]:
# ── 4.1 DateTime Feature Extraction (Day 2) ───────────────────────────────────

# Order date features
df['order_year']     = df['order_date'].dt.year
df['order_month']    = df['order_date'].dt.month
df['order_month_nm'] = df['order_date'].dt.month_name()
df['order_quarter']  = df['order_date'].dt.quarter.map({1:'Q1', 2:'Q2', 3:'Q3', 4:'Q4'})
df['order_day_name'] = df['order_date'].dt.day_name()
df['is_weekend']     = df['order_date'].dt.dayofweek >= 5    # 5=Sat, 6=Sun

# Shipping speed: days between order and ship
df['days_to_ship'] = (df['ship_date'] - df['order_date']).dt.days

print("📅 DateTime features added:")
print(df[['order_date', 'ship_date', 'order_year', 'order_month_nm', 
          'order_quarter', 'order_day_name', 'is_weekend', 'days_to_ship']].head(8))

print(f"\nAvg days to ship : {df['days_to_ship'].mean():.1f}")
print(f"Max days to ship : {df['days_to_ship'].max()}")
print(f"Weekend orders   : {df['is_weekend'].sum():,} ({df['is_weekend'].mean()*100:.1f}%)")

In [ ]:
# ── 4.2 Business Features: Lambda + Comprehensions (Day 1) ───────────────────

# Lambda: classify profit margin
profit_margin_label = lambda row: (
    'Loss'    if row['profit'] < 0
    else 'Low'    if row['profit'] / max(row['sales'], 0.01) < 0.10
    else 'Medium' if row['profit'] / max(row['sales'], 0.01) < 0.30
    else 'High'
)
df['margin_label'] = df.apply(profit_margin_label, axis=1)

# Lambda: discount tier classification
df['discount_tier'] = df['discount'].apply(
    lambda d: 'No Discount' if d == 0.0
              else 'Low (1-20%)'  if d <= 0.20
              else 'Mid (21-40%)' if d <= 0.40
              else 'High (>40%)'
)

# List comprehension: flag orders that need review
df['is_problem_order'] = [
    True if (profit < 0 and discount >= 0.3) else False
    for profit, discount in zip(df['profit'], df['discount'])
]

# Dictionary comprehension: build a ship_mode → priority map
ship_priority = {
    mode: priority
    for mode, priority in zip(
        ['Same Day', 'First Class', 'Second Class', 'Standard Class'],
        [1, 2, 3, 4]
    )
}
print(f"Ship Priority Map (dict comprehension): {ship_priority}")
df['ship_priority'] = df['ship_mode'].map(ship_priority)

print("\nNew business features:")
print(df[['order_id','sales','profit','margin_label','discount_tier','is_problem_order','ship_priority']].head(10))

In [ ]:
# ── 4.3 Day 3: Walrus Operator `:=` ──────────────────────────────────────────
#
# Real use-case: find and immediately report anomalies without two separate lines

print("=== Anomaly Detection with Walrus Operator ===")

# Check for unusually fast shipping (0 days) — same day delivery or data error?
if (same_day_count := (df['days_to_ship'] == 0).sum()) > 0:
    print(f"⚠️  Same-day shipments found: {same_day_count} orders")
    print(df[df['days_to_ship'] == 0][['order_id', 'ship_mode', 'days_to_ship']].head())
else:
    print("No same-day shipments.")

print()

# Walrus in a while-like scenario: process discount tiers one-by-one
tier_counts = df['discount_tier'].value_counts().to_dict()
tier_list   = list(tier_counts.items())
idx = 0
print("Discount tier report (walrus in while loop):")
while idx < len(tier_list) and (item := tier_list[idx]):
    tier, count = item
    print(f"   {tier:<20} : {count:>5,} orders")
    idx += 1

---
## 🔴 Section 5 — Analysis (Day 2 + Day 3: Counter, defaultdict)

### Concepts Covered
- **GroupBy** with single/multiple columns, `agg()`, `transform()`
- **Merge** — derive separate dimension tables and join back
- **Pivot Tables** — Region × Category revenue cross-view
- **Crosstab** — Segment × Ship Mode frequency
- **Multi-Index** — stacking and unstacking
- **`collections.Counter`** (Day 3)
- **`collections.defaultdict`** (Day 3)

In [ ]:
# ── 5.1 GroupBy — Category Performance ───────────────────────────────────────

category_perf = df.groupby('category').agg(
    total_orders    = ('order_id',  'count'),
    total_sales     = ('sales',     'sum'),
    total_profit    = ('profit',    'sum'),
    avg_discount    = ('discount',  'mean'),
    avg_days_ship   = ('days_to_ship', 'mean'),
    problem_orders  = ('is_problem_order', 'sum'),
).round(2)

category_perf['profit_margin_pct'] = (
    category_perf['total_profit'] / category_perf['total_sales'] * 100
).round(1)

print("📊 Category Performance Summary:")
print(category_perf.sort_values('total_sales', ascending=False))

In [ ]:
# ── 5.2 GroupBy — Region + Year Trend ─────────────────────────────────────────

region_year = df.groupby(['region', 'order_year']).agg(
    sales  = ('sales',  'sum'),
    profit = ('profit', 'sum'),
    orders = ('order_id', 'count'),
).round(2).reset_index()

print("🗺️  Region × Year Sales:")
print(region_year.sort_values(['region','order_year']))

In [ ]:
# ── 5.3 GroupBy transform — % of category total ──────────────────────────────

# transform returns same-length Series — perfect for adding a relative column
df['category_total_sales'] = df.groupby('category')['sales'].transform('sum')
df['pct_of_category']      = (df['sales'] / df['category_total_sales'] * 100).round(4)

# Also: rank within each sub_category by sales
df['sales_rank_in_subcat'] = df.groupby('sub_category')['sales'].rank(
    ascending=False, method='dense'
).astype(int)

print("Transform result — each order's % of its category total:")
print(df[['order_id','category','sales','category_total_sales','pct_of_category','sales_rank_in_subcat']].head(8))

In [ ]:
# ── 5.4 Merge — Build Dimension Tables and Join Back ────────────────────────
#
# Real-world pattern: often you receive data split across tables.
# Here we DERIVE separate dimension tables FROM the superstore data,
# then merge them back — same operations you'd do in a real ETL job.

# ── Customer dimension table ──
customers_dim = df.groupby('customer_id').agg(
    customer_name   = ('customer_name', 'first'),
    segment         = ('segment',       'first'),
    city            = ('city',          'first'),
    state           = ('state',         'first'),
    lifetime_sales  = ('sales',         'sum'),
    lifetime_profit = ('profit',        'sum'),
    total_orders    = ('order_id',      'count'),
).reset_index().round(2)

# Assign tier based on lifetime sales using a function
def customer_tier(sales: float) -> str:
    if sales >= 10000: return 'Platinum'
    if sales >= 5000:  return 'Gold'
    if sales >= 2000:  return 'Silver'
    return 'Bronze'

customers_dim['tier'] = customers_dim['lifetime_sales'].apply(customer_tier)

print(f"Customer dimension: {customers_dim.shape}")
print(customers_dim.head())
print("\nTier distribution:")
print(customers_dim['tier'].value_counts())

In [ ]:
# ── Product dimension table ──
products_dim = df.groupby('product_id').agg(
    product_name  = ('product_name',  'first'),
    category      = ('category',      'first'),
    sub_category  = ('sub_category',  'first'),
    avg_price     = ('sales',         'mean'),      # proxy for price
    times_ordered = ('order_id',      'count'),
    total_profit  = ('profit',        'sum'),
).reset_index().round(2)

print(f"Product dimension: {products_dim.shape}")
print(products_dim.head())

# LEFT JOIN: add customer tier to the main transactions
df_enriched = pd.merge(
    df,
    customers_dim[['customer_id', 'tier', 'lifetime_sales']],
    on='customer_id',
    how='left'
)

print(f"\nEnriched df shape: {df_enriched.shape}")
print(df_enriched[['order_id','customer_name','sales','tier','lifetime_sales']].head(6))

In [ ]:
# ── 5.5 Pivot Table — Region × Category Revenue ───────────────────────────────

pivot_rev = df.pivot_table(
    values   = 'sales',
    index    = 'region',
    columns  = 'category',
    aggfunc  = 'sum',
    margins  = True,
    margins_name = 'Total'
).round(0)

print("💰 Sales by Region × Category ($):")
print(pivot_rev)

print()

# Pivot with multiple aggfuncs
pivot_profit = df.pivot_table(
    values  = 'profit',
    index   = 'region',
    columns = 'category',
    aggfunc = ['sum', 'mean']
).round(1)

print("📈 Profit (sum & mean) by Region × Category:")
print(pivot_profit)

In [ ]:
# ── 5.6 Crosstab — Segment × Ship Mode ───────────────────────────────────────

ct = pd.crosstab(
    df['segment'],
    df['ship_mode'],
    margins=True,
    margins_name='Total'
)
print("📋 Order Count — Customer Segment × Ship Mode:")
print(ct)

In [ ]:
# ── 5.7 Multi-Index DataFrame — Region × Quarter ─────────────────────────────

mi_df = df.groupby(['region', 'order_quarter']).agg(
    sales  = ('sales',  'sum'),
    profit = ('profit', 'sum'),
    orders = ('order_id', 'count'),
).round(2)

print("Multi-Index DataFrame (Region × Quarter):")
print(mi_df)

# xs — cross-section: Q1 data across all regions
print("\nQ1 data across all regions (.xs):")
print(mi_df.xs('Q1', level='order_quarter'))

# unstack — pivot inner index (quarter) to columns
print("\nUnstacked (quarters as columns):")
print(mi_df['sales'].unstack(level='order_quarter').round(0))

In [ ]:
# ── 5.8 Day 3: Counter & defaultdict ─────────────────────────────────────────

# Counter: most common sub-categories ordered
subcat_counter = Counter(df['sub_category'])
print("🛒 Top 10 Most Ordered Sub-Categories (Counter):")
for subcat, count in subcat_counter.most_common(10):
    bar = '█' * (count // 20)
    print(f"   {subcat:<20} {count:>5}  {bar}")

print()

# defaultdict: group product names by category without pandas
products_by_cat = defaultdict(list)
for _, row in products_dim.iterrows():
    products_by_cat[row['category']].append(row['product_name'])

print("📦 Product counts by category (defaultdict):")
for cat, prods in products_by_cat.items():
    print(f"   {cat:<20} : {len(prods)} unique products")

---
## 🟣 Section 6 — OOP Pipeline (Day 1 + Day 3: Decorators, `logging`, `@property`)

### Concepts Covered
- **Class variables** — shared across instances
- **`__init__`** — constructor and instance variables
- **Instance methods** — `.extract()`, `.transform()`, `.load()`
- **`@property`** (Day 3) — computed read-only attributes
- **`@timer` decorator** (Day 3) — functools.wraps, execution timing
- **`@log_step` decorator factory** (Day 3) — parameterised decorator
- **Inheritance** — `ReportPipeline` extends `SuperstorePipeline`
- **`logging` module** (Day 3) — production logs

In [ ]:
# ── 6.1 Logging Setup (Day 3) ─────────────────────────────────────────────────

logging.basicConfig(
    level   = logging.INFO,
    format  = '%(asctime)s | %(levelname)-8s | %(name)s | %(message)s',
    datefmt = '%H:%M:%S'
)
logger = logging.getLogger('SuperstorePipeline')
logger.info("Logger initialised ✅")

In [ ]:
# ── 6.2 Decorators: @timer and @log_step factory (Day 3) ────────────────────

def timer(func):
    """
    Decorator — measures wall-clock execution time of any function.
    Logs the result at INFO level.
    Uses functools.wraps to preserve original function metadata.
    """
    @wraps(func)                          # preserves __name__, __doc__
    def wrapper(*args, **kwargs):
        t0     = time.perf_counter()
        result = func(*args, **kwargs)
        t1     = time.perf_counter()
        logger.info(f"⏱  {func.__name__}() → {t1 - t0:.4f}s")
        return result
    return wrapper


def log_step(step_name: str):
    """
    Decorator FACTORY — takes a configuration argument (step_name)
    and returns a decorator. This is the pattern behind @retry(n=3),
    @app.get('/route'), etc.
    """
    def decorator(func):           # actual decorator (takes a function)
        @wraps(func)
        def wrapper(*args, **kwargs):
            logger.info(f"▶  START  : {step_name}")
            try:
                result = func(*args, **kwargs)
                logger.info(f"✅ DONE   : {step_name}")
                return result
            except Exception as exc:
                logger.error(f"❌ FAILED : {step_name} — {exc}")
                raise
        return wrapper
    return decorator   # decorator factory returns the decorator


# Verify @wraps preserved function metadata
@timer
def test_timer():
    """A test function."""
    time.sleep(0.05)

test_timer()
print(f"Function name after @timer: '{test_timer.__name__}' (not 'wrapper' — thanks to @wraps!)")
print(f"Docstring preserved       : '{test_timer.__doc__}'")

In [ ]:
# ── 6.3 Base + Derived Pipeline Classes (Day 1 OOP + Day 3 Decorators) ───────

class BasePipeline:
    """Abstract base class for data pipelines."""

    source_name = "Kaggle Superstore Dataset"  # Class variable — shared by all instances

    def __init__(self, filepath: str, output_dir: str):
        self.filepath   = Path(filepath)
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        self._df_raw    = None     # underscore = private by convention
        self._df_clean  = None
        self._run_ts    = None
        self.metrics    = {}

    # ── @property (Day 3) — computed read-only attributes ─────────────────
    @property
    def row_count(self) -> int:
        """Read-only: number of rows in cleaned DataFrame."""
        return len(self._df_clean) if self._df_clean is not None else 0

    @property
    def is_ready(self) -> bool:
        """Read-only: True once transform() has been called."""
        return self._df_clean is not None

    @property
    def columns(self):
        """Read-only: list of cleaned DataFrame columns."""
        return list(self._df_clean.columns) if self._df_clean is not None else []

    def describe(self):
        return (
            f"[{self.source_name}] "
            f"File: {self.filepath.name} | "
            f"Ready: {self.is_ready} | "
            f"Rows: {self.row_count:,}"
        )


# ─────────────────────────────────────────────────────────────────────────────

class SuperstorePipeline(BasePipeline):
    """
    ETL pipeline for the Kaggle Superstore dataset.
    Inherits from BasePipeline.
    Each method is decorated with @timer and @log_step.
    Methods return self to support method chaining.
    """

    @timer
    @log_step('Extract — Load CSV')
    def extract(self) -> 'SuperstorePipeline':
        """Load raw CSV. Tries multiple encodings."""
        for enc in ['utf-8', 'latin-1', 'cp1252']:
            try:
                self._df_raw = pd.read_csv(self.filepath, encoding=enc)
                self.metrics['encoding']    = enc
                self.metrics['raw_rows']    = len(self._df_raw)
                self.metrics['raw_cols']    = len(self._df_raw.columns)
                return self
            except UnicodeDecodeError:
                continue
        raise IOError(f"Cannot read {self.filepath}")

    @timer
    @log_step('Transform — Clean & Engineer')
    def transform(self) -> 'SuperstorePipeline':
        """Clean types, deduplicate, engineer key features."""
        df = self._df_raw.rename(columns=COL_RENAME).drop_duplicates()

        # Type fixes
        for col in ['order_date', 'ship_date']:
            df[col] = pd.to_datetime(df[col], infer_datetime_format=True, errors='coerce')
        df['sales']    = pd.to_numeric(df['sales'],    errors='coerce')
        df['profit']   = pd.to_numeric(df['profit'],   errors='coerce')
        df['discount'] = pd.to_numeric(df['discount'], errors='coerce')
        df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce').astype('Int64')
        df = df.dropna(subset=['order_date', 'sales'])

        # DateTime features
        df['order_year']    = df['order_date'].dt.year
        df['order_quarter'] = df['order_date'].dt.quarter.map({1:'Q1',2:'Q2',3:'Q3',4:'Q4'})
        df['order_month']   = df['order_date'].dt.month
        df['days_to_ship']  = (df['ship_date'] - df['order_date']).dt.days

        # Business features
        df['discount_tier']    = df['discount'].apply(
            lambda d: 'None' if d==0 else ('Low' if d<=0.2 else ('Mid' if d<=0.4 else 'High'))
        )
        df['is_problem_order'] = (df['profit'] < 0) & (df['discount'] >= 0.3)
        df['margin_pct']       = (df['profit'] / df['sales'].replace(0, np.nan) * 100).round(2)

        self._df_clean = df
        self.metrics['clean_rows'] = len(df)
        self.metrics['removed']    = self.metrics['raw_rows'] - len(df)
        return self

    @timer
    @log_step('Load — Save to Disk')
    def load(self, fmt: str = 'csv') -> 'SuperstorePipeline':
        """Export cleaned data. fmt in {'csv', 'excel', 'parquet'}."""
        if not self.is_ready:
            raise RuntimeError("Call .transform() before .load()")

        df = self._df_clean.copy()
        # Serialise datetimes for csv/excel
        df['order_date'] = df['order_date'].dt.strftime('%Y-%m-%d')
        df['ship_date']  = df['ship_date'].dt.strftime('%Y-%m-%d')

        paths = {'csv': 'clean.csv', 'excel': 'clean.xlsx', 'parquet': 'clean.parquet'}
        if fmt not in paths:
            raise ValueError(f"fmt must be one of {list(paths.keys())}")

        out = self.output_dir / paths[fmt]
        if fmt == 'csv':     df.to_csv(out, index=False)
        elif fmt == 'excel': df.to_excel(out, index=False, sheet_name='Superstore')
        elif fmt == 'parquet': self._df_clean.to_parquet(out, index=False)

        self._run_ts = datetime.now()
        logger.info(f"   Saved → {out}")
        return self

    def summary(self) -> dict:
        return {**self.metrics, 'run_at': self._run_ts, 'row_count': self.row_count}


print("Classes defined ✅")

In [ ]:
# ── 6.4 Run the Pipeline ──────────────────────────────────────────────────────

pipeline = SuperstorePipeline(
    filepath   = FILE_PATH,
    output_dir = OUTPUT_DIR
)

print(pipeline.describe())       # before run
print(f"is_ready : {pipeline.is_ready}")
print()

# Method chaining — each step returns self
pipeline.extract().transform().load(fmt='csv')

print()
print(pipeline.describe())       # after run
print(f"Columns  : {pipeline.columns[:8]} ...")
print()
print("Pipeline Summary:")
for k, v in pipeline.summary().items():
    print(f"   {k:<20}: {v}")

In [ ]:
# ── 6.5 Inheritance: ReportPipeline extends SuperstorePipeline ───────────────

class ReportPipeline(SuperstorePipeline):
    """
    Extends SuperstorePipeline by adding multi-sheet Excel report generation.
    Demonstrates inheritance: gets all ETL methods from parent for free.
    """

    def __init__(self, filepath, output_dir='superstore_output'):
        super().__init__(filepath, output_dir)   # call parent constructor
        self.reports = {}                        # new attribute — only in child

    @timer
    @log_step('Generate Excel Report')
    def generate_report(self) -> 'ReportPipeline':
        """Build and export summary reports to a multi-sheet Excel file."""
        df = self._df_clean

        self.reports['category_summary'] = df.groupby('category').agg(
            orders  = ('order_id', 'count'),
            sales   = ('sales',    'sum'),
            profit  = ('profit',   'sum'),
        ).round(2)

        self.reports['region_summary'] = df.groupby('region').agg(
            orders  = ('order_id', 'count'),
            sales   = ('sales',    'sum'),
            profit  = ('profit',   'sum'),
        ).round(2)

        self.reports['sub_category_ranking'] = (
            df.groupby('sub_category')['profit'].sum()
              .sort_values(ascending=False)
              .reset_index()
              .rename(columns={'profit': 'total_profit'})
        )

        self.reports['top_customers'] = (
            df.groupby(['customer_id', 'customer_name', 'segment'])
              .agg(orders=('order_id','count'), sales=('sales','sum'), profit=('profit','sum'))
              .sort_values('sales', ascending=False)
              .head(20).round(2)
        )

        # Write all reports to one Excel file (multi-sheet)
        excel_path = self.output_dir / 'superstore_report.xlsx'
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            for sheet, report_df in self.reports.items():
                report_df.to_excel(writer, sheet_name=sheet)

        logger.info(f"   Saved report → {excel_path}")
        return self


# Run full pipeline including report generation
print("=" * 55)
report_p = ReportPipeline(FILE_PATH)
report_p.extract().transform().load(fmt='parquet').generate_report()

print()
print("📊 Category Summary (from ReportPipeline):")
print(report_p.reports['category_summary'])

print()
print("🏆 Top 5 Sub-Categories by Profit:")
print(report_p.reports['sub_category_ranking'].head())

---
## 🟤 Section 7 — Visualisation (Day 3: 5 Chart Types)

### Concepts Covered
- **Line Chart** — annual monthly sales trend
- **Bar Chart** — category comparison with value labels
- **Scatter Plot** — discount vs profit, colour-coded by category
- **Histogram** — order value distribution
- **Subplots (2×2 Dashboard)** — executive summary view
- **`savefig()`** — export to PNG

In [ ]:
# ── 7.1 Line Chart — Monthly Sales Trend ─────────────────────────────────────

monthly = df.groupby(['order_year', 'order_month'])['sales'].sum().reset_index()
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(13, 4))

colors_yr = {'2014':'#AED6F1', '2015':'#5DADE2', '2016':'#2874A6', '2017':'#1B2631'}

for year in sorted(monthly['order_year'].unique()):
    yr_data = monthly[monthly['order_year'] == year].sort_values('order_month')
    ax.plot(
        yr_data['order_month'], yr_data['sales'],
        marker='o', linewidth=2, markersize=6,
        label=str(year), color=colors_yr.get(str(year), 'grey')
    )

ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.set_title('Monthly Sales Trend by Year', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Total Sales ($)')
ax.legend(title='Year')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'chart_monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Line chart saved")

In [ ]:
# ── 7.2 Bar Chart — Sub-Category Profit (sorted) ─────────────────────────────

subcat_profit = df.groupby('sub_category')['profit'].sum().sort_values()
colors_bar    = ['tomato' if v < 0 else 'steelblue' for v in subcat_profit]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(subcat_profit.index, subcat_profit.values,
               color=colors_bar, edgecolor='white', linewidth=0.6)

ax.axvline(0, color='black', linewidth=0.8, linestyle='--')

for bar in bars:
    w = bar.get_width()
    ax.text(w + (500 if w >= 0 else -500),
            bar.get_y() + bar.get_height() / 2,
            f'${w:,.0f}', va='center', ha='left' if w >= 0 else 'right', fontsize=8)

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.set_title('Total Profit by Sub-Category\n(Red = Losing Money)', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Profit ($)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'chart_subcat_profit.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Bar chart saved")

In [ ]:
# ── 7.3 Scatter Plot — Discount vs Profit ────────────────────────────────────

cat_colors = {'Furniture':'#FF9800', 'Office Supplies':'#4CAF50', 'Technology':'#2196F3'}

fig, ax = plt.subplots(figsize=(9, 6))

for cat, col in cat_colors.items():
    subset = df[df['category'] == cat]
    ax.scatter(subset['discount'] * 100, subset['profit'],
               alpha=0.4, s=30, color=col, label=cat)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--', label='Break-even')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_title('Discount % vs Profit per Order', fontsize=14, fontweight='bold')
ax.set_xlabel('Discount (%)')
ax.set_ylabel('Profit ($)')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'chart_discount_profit.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Scatter plot saved")

In [ ]:
# ── 7.4 Histogram — Order Sales Distribution ─────────────────────────────────

fig, ax = plt.subplots(figsize=(9, 4))

# Clip to 99th percentile so extreme outliers don't squash the chart
cap = df['sales'].quantile(0.99)
ax.hist(df['sales'].clip(upper=cap), bins=60, color='steelblue',
        edgecolor='white', alpha=0.85)

mean_s   = df['sales'].mean()
median_s = df['sales'].median()
ax.axvline(mean_s,   color='tomato',    linestyle='--', linewidth=2, label=f'Mean  ${mean_s:,.0f}')
ax.axvline(median_s, color='limegreen', linestyle='--', linewidth=2, label=f'Median ${median_s:,.0f}')

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_title(f'Order Sales Distribution (capped at 99th percentile: ${cap:,.0f})',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Sales per Order ($)')
ax.set_ylabel('Number of Orders')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'chart_sales_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Histogram saved")

In [ ]:
# ── 7.5 Subplots (2×2) — Executive Dashboard ─────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Superstore Executive Dashboard', fontsize=16, fontweight='bold', y=1.01)

# ── Top-left: Sales by Category (Bar) ──
ax = axes[0, 0]
cat_s = df.groupby('category')['sales'].sum().sort_values()
ax.barh(cat_s.index, cat_s.values, color=['#FF9800','#4CAF50','#2196F3'])
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
ax.set_title('Total Sales by Category')

# ── Top-right: Region Profit (Bar) ──
ax = axes[0, 1]
reg_p = df.groupby('region')['profit'].sum().sort_values()
colors_reg = ['tomato' if v < 0 else 'steelblue' for v in reg_p]
ax.barh(reg_p.index, reg_p.values, color=colors_reg)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Total Profit by Region')

# ── Bottom-left: Customer Segment Pie ──
ax = axes[1, 0]
seg = df['segment'].value_counts()
ax.pie(seg.values, labels=seg.index, autopct='%1.1f%%', startangle=90,
       colors=['#2196F3','#4CAF50','#FF9800'])
ax.set_title('Orders by Customer Segment')

# ── Bottom-right: Quarterly Sales Line ──
ax = axes[1, 1]
q_sales = df.groupby(['order_year', 'order_quarter'])['sales'].sum().reset_index()
q_sales['period'] = q_sales['order_year'].astype(str) + ' ' + q_sales['order_quarter']
ax.plot(range(len(q_sales)), q_sales['sales'], marker='o', color='steelblue', linewidth=2)
ax.set_xticks(range(len(q_sales)))
ax.set_xticklabels(q_sales['period'], rotation=45, ha='right', fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.set_title('Quarterly Sales Trend')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Dashboard saved")

---
## ⚡ Section 8 — Export & Generators (Day 2 + Day 3)

### Concepts Covered
- **Export to CSV, Excel (multi-sheet), Parquet** (Day 2)
- **Generator function** (Day 3) — batch export, memory-efficient
- **Generator expression** (Day 3) — vs list comprehension memory comparison
- **`itertools.islice`** (Day 3) — lazy first-N from any iterable

In [ ]:
# ── 8.1 Multi-Format Export (Day 2) ──────────────────────────────────────────

# CSV
df.to_csv(OUTPUT_DIR / 'superstore_clean.csv', index=False)
print("✅ CSV exported")

# Parquet — columnar format, ~3–5× smaller than CSV, much faster to read
df.to_parquet(OUTPUT_DIR / 'superstore_clean.parquet', index=False)
print("✅ Parquet exported")

# Excel — 4 sheets in one file
with pd.ExcelWriter(OUTPUT_DIR / 'superstore_analysis.xlsx', engine='openpyxl') as writer:

    # Sheet 1: All data
    df.to_excel(writer, sheet_name='All_Orders', index=False)

    # Sheet 2: Category summary
    df.groupby('category').agg(
        orders=('order_id','count'), sales=('sales','sum'), profit=('profit','sum')
    ).round(2).to_excel(writer, sheet_name='Category_Summary')

    # Sheet 3: Region × Quarter pivot
    df.pivot_table(
        values='sales', index='region', columns='order_quarter', aggfunc='sum'
    ).round(0).to_excel(writer, sheet_name='Regional_Pivot')

    # Sheet 4: Loss-making orders
    df[df['profit'] < 0].sort_values('profit')[[
        'order_id','customer_name','product_name','sales','discount','profit'
    ]].head(100).to_excel(writer, sheet_name='Problem_Orders', index=False)

print("✅ Excel exported (4 sheets)")

# Compare CSV vs Parquet file sizes
csv_size     = (OUTPUT_DIR / 'superstore_clean.csv').stat().st_size / 1024
parquet_size = (OUTPUT_DIR / 'superstore_clean.parquet').stat().st_size / 1024
print(f"\n📦 CSV size     : {csv_size:>8.1f} KB")
print(f"📦 Parquet size : {parquet_size:>8.1f} KB  ({csv_size/parquet_size:.1f}× smaller!)")

In [ ]:
# ── 8.2 Generator vs List — Memory Comparison (Day 3) ────────────────────────
#
# Key teaching point: generators don't hold values in memory — they compute on demand.

# List comprehension — ALL values in memory at once
sales_list = [round(s, 2) for s in df['sales']]

# Generator expression — computes one value at a time (lazy)
sales_gen  = (round(s, 2) for s in df['sales'])

print("Memory comparison:")
print(f"  List  size  : {sys.getsizeof(sales_list):>10,} bytes  ← stores ALL values")
print(f"  Generator   : {sys.getsizeof(sales_gen):>10,} bytes  ← stores just the recipe!")
print()
print("Both produce the same output when iterated, but the generator")
print("uses only ~120 bytes regardless of how many rows your data has.")
print("This is critical for processing millions of rows in a real pipeline.")

In [ ]:
# ── 8.3 Generator Function: Batch Export (Day 3) ────────────────────────────
#
# Real use-case: your cleaned data is 10M rows. You can't load it all in RAM.
# A generator writes one batch at a time — RAM usage stays flat.

def batch_writer(dataframe: pd.DataFrame, batch_size: int, output_folder: Path):
    """
    Generator function — yields (batch_number, file_path) for each batch written.
    Uses 'yield' not 'return': execution pauses at yield and resumes on next().
    """
    total   = len(dataframe)
    batches = range(0, total, batch_size)

    for batch_num, start in enumerate(batches, start=1):
        end      = min(start + batch_size, total)
        batch_df = dataframe.iloc[start:end]
        path     = output_folder / f'batch_{batch_num:03d}.csv'
        batch_df.to_csv(path, index=False)
        yield batch_num, path      # ← pauses here, returns value, resumes next iteration


# Use the generator — only one batch in memory at any time
BATCH_SIZE  = 2000
batch_dir   = OUTPUT_DIR / 'batches'
batch_dir.mkdir(exist_ok=True)

print(f"Writing {len(df):,} rows in batches of {BATCH_SIZE}...")
batch_results = []

for num, path in batch_writer(df, BATCH_SIZE, batch_dir):
    rows = pd.read_csv(path).shape[0]
    print(f"   Batch {num:02d}: {rows:>5,} rows → {path.name}")
    batch_results.append(path)

print(f"\n✅ {len(batch_results)} batch files written")

In [ ]:
# ── 8.4 itertools.islice + chain — Read Back First 2 Batches (Day 3) ─────────

# itertools.islice: lazily take first N items from any iterable
first_two_paths = list(itertools.islice(sorted(batch_dir.glob('*.csv')), 2))
print(f"First 2 batch files: {[p.name for p in first_two_paths]}")

# itertools.chain — flatten multiple DataFrames into one
# (Equivalent to pd.concat but shows how chain works conceptually)
chunk_dfs = [pd.read_csv(p) for p in first_two_paths]
combined  = pd.concat(chunk_dfs, ignore_index=True)
print(f"Combined first 2 batches: {combined.shape[0]:,} rows (expected {min(2*BATCH_SIZE, len(df)):,})")

# itertools.chain on raw values
category_lists  = [['Furniture'], ['Office Supplies'], ['Technology']]
flat_categories = list(itertools.chain.from_iterable(category_lists))
print(f"\nchain.from_iterable demo: {flat_categories}")

---
## 🏁 Section 9 — Summary & Output Files

Every concept from Days 1, 2 and 3 used on **real Kaggle data**:

```
Sample - Superstore.csv  (Kaggle raw data)
        │
        ▼
Section 1  ─── Variables, Tuples, Dicts, Custom Exception, pathlib
        │
        ▼
Section 2  ─── head/info/describe, loc/iloc, sort_values, isin/filters
        │
        ▼
Section 3  ─── Column rename, Null check, Dedup, astype/to_datetime, re module
        │
        ▼
Section 4  ─── DateTime extraction, Lambda, List/Dict comprehension, Walrus :=
        │
        ▼
Section 5  ─── GroupBy+agg/transform, Merge, Pivot, Crosstab, MultiIndex,
               Counter, defaultdict
        │
        ▼
Section 6  ─── OOP (Class/Inheritance/@property), @timer, @log_step factory,
               functools.wraps, logging module
        │
        ▼
Section 7  ─── Line, Bar, Scatter, Histogram, Subplots Dashboard, savefig
        │
        ▼
Section 8  ─── CSV/Excel/Parquet export, Generator vs List, batch_writer
               generator function, itertools.islice + chain
```

In [ ]:
# ── Final File Inventory ──────────────────────────────────────────────────────

print("📁 All files generated:\n")
for folder in [OUTPUT_DIR, OUTPUT_DIR / 'batches']:
    if folder.exists():
        files = sorted(folder.glob('*'))
        if files:
            print(f"📂 {folder}/")
            for f in files:
                if f.is_file():
                    ext  = f.suffix
                    icon = {'csv':'📄','.xlsx':'📊','.parquet':'🗃️','.png':'🖼️'}.get(ext, '📄')
                    size = f.stat().st_size / 1024
                    print(f"   {icon} {f.name:<45} {size:>7.1f} KB")
            print()

print("\n🎉 Project complete!")

---
## 📝 Practice Exercises

Use the real Superstore data to complete these:

### Exercise 1 — Day 1 (Easy)
Write a function `format_currency(amount, symbol='$')` using default parameters and an f-string that returns `$1,234.56` format. Apply it to the `sales` column using `.apply()`.

### Exercise 2 — Day 2 (Moderate)
Using the `df`, find the **top 5 customers** who have placed the most orders in the **West region** using `GroupBy`, `sort_values`, and `head()`.

### Exercise 3 — Day 2 + Day 1 (Moderate)
Build a Multi-Index DataFrame indexed by `['segment', 'order_year']`. Use `.xs()` to extract all **Corporate** orders from 2017. Then use `.unstack()` to see years as columns.

### Exercise 4 — Day 3: Decorator (Challenging)
Write a `@memoize` decorator that caches the result of a function call. Apply it to an expensive aggregation function like:
```python
@memoize
def get_region_total(region_name):
    return df[df['region'] == region_name]['sales'].sum()
```
Call it twice for the same region — the second call should be instant.

### Exercise 5 — Day 3: Generator (Challenging)
Write a generator function `loss_order_stream(df)` that yields orders with negative profit **one at a time** — don't build a list. Print the first 10 yielded orders using `itertools.islice`.

### Exercise 6 — Day 1+2: OOP (Challenging)
Add a `validate()` method to `SuperstorePipeline` that:
- Raises a custom `DataQualityError` if more than 30% of orders are loss-making
- Raises it if any `sales` value is negative
- Returns `self` (for chaining) if everything passes
- Is decorated with `@log_step('Data Validation')`